# 0 - Download Remote Sensing Data
**Purpose:** Download different Types of RS data which could be useful for the RiverEggCode.

**Stuff:**

0 | Imports

0.1 | (SWOT) Surface Water and Ocean Topography
- 0.1.1 | ....
- 0.1.2 | ...

To-Do:
- []

---
---
## 0 | Imports


In [27]:
import geopandas as gpd
import pandas as pd
import requests
import json
import os
import sys
import time
import zipfile
from pathlib import Path

import earthaccess

sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT
from _0_download_rs import query_hydrocron_reach

---
---
## 0.1 | (SWOT) Surface Water and Ocean Topography 

**Purpose:** Download NASA SWOT hydrology data for a defined AOI, without downloading
the full global datasets. Two approaches are implemented:

- **0.1.1 | Hydrocron API**: Pulls SWOT time series for individual SWORD reaches already
  clipped to the AOI (Notebook 01 output). No file download, no NASA login required.
  This is to to be able to assess the data situation and figure out what variables could be derived or come up with from it, and so on. 

- **0.1.2 | earthaccess**: Searches and downloads the actual SWOT Shapefile granules
  (e.g. RiverSP/ Raster) filtered by bounding box and time range. Files are
  downloaded locally and clipped to the AOI. NASA Earthdata login required.
  Best for full spatial coverage, raster products but HUGEEEE amount of data.

**To-Do:**

- [] How should can I besser handle the amount of AWOT data? Maybe...by..maybe creating a Goal-Reach_list and using parquet instead of shp and gpkg.

**References:**
- SWOT Mission: (https://swot.jpl.nasa.gov)
- PO.DAAC SWOT Data: (https://podaac.jpl.nasa.gov/SWOT)
- Tebaldi, N., Nickels, C. (2025): [*Hydrocron API: SWOT Time Series Examples.*](https://podaac.github.io/tutorials/notebooks/datasets/Hydrocron_SWOT_timeseries_examples.html)
- Nickels, C. (2025): [*Search and Download SWOT Data via <code>earthaccess</code>*. ](https://podaac.github.io/tutorials/notebooks/SearchDownload_SWOTviaCMR.html)
- earthaccess docs: (https://earthaccess.readthedocs.io)
- NASA Earthdata Login: (https://urs.earthdata.nasa.gov)



In [32]:
# CONFIGURATION
# INPUT: clipped SWORD from Notebook 01 
IN_SWORD_CLIPPED = os.path.join(DATA_PROCESSED, "sword_naryn_clip.gpkg")

#--- TIME RANGE for SWOT observations -------------------------------------
# Format: ISO 8601 (YYYY-MM-DDTHH:MM:SSZ)
START_TIME = "2025-08-01T00:00:00Z" # Earliest date: 2023-08-01T00:00:00Z
END_TIME = "2025-08-31T00:00:00Z"


#--- HYDROCRON: which SWOT fields to retrieve per reach --------------------
# NOTE: For full list of fields look in "_0_download_rs.py" file
HYDROCRON_FIELDS = "reach_id,time_str,wse ,slope ,width ,p_dist_out ,geometry"
HYDROCRON_OUTPUT_FORMAT = "geojson" # "geojson" or "csv"

# Delay between API calls
HYDROCRON_REQUEST_DELAY = 0.3

#--- EARTHACCESS: which product collection to download ---------------------
# Most relevant collections:
#   SWOT_L2_HR_RiverSP_reach_D: River reaches (Shapefile, ~10 km segments)
#   SWOT_L2_HR_RiverSP_node_D: River nodes (Shapefile, ~200 m spacing)
#   SWOT_L2_HR_Raster_D: Raster WSE/water mask (netCDF, large files)

# NOTE: "_D" suffix = Version D (latest as of 2025). Change if newer version exists.
EARTHACCESS_COLLECTION = "SWOT_L2_HR_RiverSP_reach_D"

#--- AOI bounding box -----------------------------------------------------
# for earthaccess spatial search
# Format: (min_lon, min_lat, max_lon, max_lat)
# Will be computed automatically from SWORD clip if left as None.
# Set manually to override, e.g. for a larger search area: e.g. BBOX = (70.0, 40.0, 80.0, 43.0)
BBOX = None

#--- OUTPUT paths ---------------------------------------------------------

OUT_SWOT_HYDROCRON = os.path.join(DATA_PROCESSED, "swot_hydrocron_naryn.gpkg")
OUT_SWOT_EARTHACCESS_DIR = os.path.join(DATA_RAW, "0_data", "SWOT_downloaded")

In [33]:
# Load SWORD clip and derive make box
sword = gpd.read_file(IN_SWORD_CLIPPED)
print(f"SWORD reaches loaded: {len(sword)}")
print(f"CRS: {sword.crs}")

# Autocompute BBOX if not set manually
if BBOX is None:
    bounds = sword.total_bounds
    # Small padding to avoid edge effects
    PAD = 0.05
    BBOX = (
        bounds[0] - PAD, # min_lon
        bounds[1] - PAD, # min_lat
        bounds[2] + PAD, # max_lon
        bounds[3] + PAD  # max_lat
    )
    print(f"\nAuto-computed BBOX (with {PAD}° padding):")
else:
    print("\nManual BBOX used:")
print(f"  (lon_min={BBOX[0]:.4f}, lat_min={BBOX[1]:.4f}, "
      f"lon_max={BBOX[2]:.4f}, lat_max={BBOX[3]:.4f})")

# Extract all reach_ids from the clipped SWORD
reach_ids = sword["reach_id"].dropna().astype(str).tolist()
print(f"\nReach IDs available: {len(reach_ids)}")
print(f"Examples: {reach_ids[:5]}")

SWORD reaches loaded: 140
CRS: EPSG:4326

Auto-computed BBOX (with 0.05° padding):
  (lon_min=71.7047, lat_min=40.8544, lon_max=78.2982, lat_max=42.3658)

Reach IDs available: 140
Examples: ['46219400056', '46219400041', '46219400021', '46219300051', '46219300061']


---
---
### 0.1.1 | Hydrocron

Needing the **Hydrocron API** (PO.DAAC / NASA JPL) for each SWORD reach in the AOI.
Returns all SWOT observations within the configured time range as GeoJSON or CSV.

**How:** SWOT data is archived as globally-tiled shapefiles per satellite pass
(one file = one pass over a whole continent). Hydrocron unpacks these shapefiles and
re-indexes them by reach_id, so a single API call returns all observations for one reach
across all passes --> without downloading any file. **No NASA login required.**

https://github.com/podaac/hydrocron

**OUTPUT**:

Referenced to WGS84-Ellipsoid with geoidcorrection: Observation from 08.2023 to current of 
- *water surface elevation* (<code>wse</code> [m])
- *water surface slope* (<code>slope</code> [m/m](calculated by wse-difference between beginning- and end-reach))
- *river width* (<code>width</code> [m](Observed SWOT swath width at the time of acquisition, derived from the water mask classification of the radar image.))
- *prior distance to outlet* (<code>p_dist_out</code> [m] (is not chaning between aquisitions. Distance between reach-meanpoint and outlet))

In [34]:
# Check reach_id format (Hydrocron expects 11-digit SWORD reach IDs):
id_lengths = sword["reach_id"].astype(str).str.len().value_counts()
print("Reach ID digit-length distribution:")
print(id_lengths.to_string())
non_11 = sword[sword["reach_id"].astype(str).str.len() != 11]
if not non_11.empty:
    print(f"\n{len(non_11)} reach_ids are NOT 11 digits -_> these will likely return Error 400.")
    print(non_11[["reach_id"]].head())
else:
    print("\nAll reach_ids are 11 digits.")

# Check SWOT width coverage:
# SWOT observes rivers wider than ca. 100 m.
# Reaches below this threshold will return no data from Hydrocron.
if "width" in sword.columns:
    width_stats = sword["width"].describe()
    n_below_100 = (sword["width"] < 100).sum()
    n_above_100 = (sword["width"] >= 100).sum()

    print(f"\nWidth statistics (SWORD, in meters):")
    print(width_stats.round(1).to_string())
    print(f"\nReaches < 100 m wide (likely no SWOT obs): {n_below_100} / {len(sword)}")
    print(f"Reaches >= 100 m wide (potential SWOT obs): {n_above_100} / {len(sword)}")
    if n_above_100 == 0:
        print("\nWARNING: No reaches exceed 100 m width. Hydrocron will return 400/no_data for most or all reaches.")
else:
    print("\nColumn 'width' not found in SWORD --> cannot estimate SWOT coverage.")

# Test the first reachable reach:
# Try the widest reach first --> more likely to have SWOT observations.
if "width" in sword.columns:
    test_row = sword.sort_values("width", ascending=False).iloc[0]
    TEST_REACH = str(test_row["reach_id"])
    print(f"\nTesting widest reach: {TEST_REACH} (width = {test_row['width']:.1f} m)")
else:
    TEST_REACH = reach_ids[0]
    print(f"\nTesting first reach: {TEST_REACH}")

result = query_hydrocron_reach(
    reach_id = TEST_REACH,
    start_time= START_TIME,
    end_time= END_TIME,
    fields= HYDROCRON_FIELDS,
    output_format = HYDROCRON_OUTPUT_FORMAT
)

print(f"\nAPI status: {result['status']}")
print(f"Message: {result['message']}")

if result["status"] == "ok":
    body = result["data"]
    print(f"Response keys: {list(body.keys())}")
    try:
        features = body["results"]["geojson"]["features"]
        print(f"Observations returned: {len(features)}")
        if features:
            print(f"\nAvailable properties: {list(features[0]['properties'].keys())}")
            print("\nSample (first obs):")
            for k, v in features[0]["properties"].items():
                print(f" {k}: {v}")
    except KeyError as e:
        print(f"Unexpected response structure. Key: {e}")
        print(json.dumps(body, indent=2)[:500])

elif result["status"] == "no_data":
    print("\nNo SWOT observations found for this reach.")

elif result["status"] == "error":
    print("\nWARNING! API error. Possible causes:")
    print("- reach_id not in Hydrocron database (SWORD version mismatch)")
    print("- invalid field name in HYDROCRON_FIELDS")
    print("- API temporarily unavailable")


Reach ID digit-length distribution:
reach_id
11    140

All reach_ids are 11 digits.

Width statistics (SWORD, in meters):
count      140.0
mean       300.8
std       1157.3
min         30.0
25%         54.0
50%         80.0
75%        151.1
max      11533.0

Reaches < 100 m wide (likely no SWOT obs): 93 / 140
Reaches >= 100 m wide (potential SWOT obs): 47 / 140

Testing widest reach: 46219100213 (width = 11533.0 m)

API status: ok
Message: success
Response keys: ['status', 'time', 'hits', 'results']
Observations returned: 4

Available properties: ['reach_id', 'time_str', 'wse', 'slope', 'width', 'p_dist_out', 'wse_units', 'slope_units', 'width_units', 'p_dist_out_units']

Sample (first obs):
 reach_id: 46219100213
 time_str: 2025-08-03T16:29:31Z
 wse: 867.099
 slope: -999999999999.0
 width: -999999999999.0
 p_dist_out: 2370287.607
 wse_units: m
 slope_units: m/m
 width_units: m
 p_dist_out_units: m


In [35]:
# HYDROCRON
# bulk query for all AOI reaches: Skips reaches narrower than MIN_WIDTH_M (<100m, likely no SWOT obs).
# Increase HYDROCRON_REQUEST_DELAY if getting repeated errors.
# Set to 0 to query all reaches regardless of width
MIN_WIDTH_M = 100

if "width" in sword.columns and MIN_WIDTH_M > 0:
    query_reaches = sword[sword["width"] >= MIN_WIDTH_M]["reach_id"].astype(str).tolist()
    skipped = len(reach_ids) - len(query_reaches)
    print(f"Reaches queried (width >= {MIN_WIDTH_M} m): {len(query_reaches)}")
    print(f"Skipped (too narrow): {skipped}")
else:
    query_reaches = reach_ids
    print(f"Querying all {len(query_reaches)} reaches (no width filter applied).")

all_features = []
n_ok = 0
n_no_data = 0
n_error = 0
error_log = []  # keep track of unexpected errors for review

print(f"\nTime range: {START_TIME} -> {END_TIME}")
for i, rid in enumerate(query_reaches):
    if i % 10 == 0:
        print(f"{i:3d}/{len(query_reaches)} | ok={n_ok} no_data={n_no_data} error={n_error}")

# Function in "_0_download_rs.py"
    result = query_hydrocron_reach(
        reach_id = rid,
        start_time = START_TIME,
        end_time = END_TIME,
        fields = HYDROCRON_FIELDS,
        output_format = HYDROCRON_OUTPUT_FORMAT
    )

    if result["status"] == "ok":
        try:
            features = result["data"]["results"]["geojson"]["features"]
            if features:
                all_features.extend(features)
                n_ok += 1
            else:
                n_no_data += 1
        except (KeyError, TypeError):
            n_error += 1
            error_log.append((rid, "unexpected response structure"))

    elif result["status"] == "no_data":
        n_no_data += 1

    else:
        n_error += 1
        error_log.append((rid, result["message"]))

    time.sleep(HYDROCRON_REQUEST_DELAY)

print("-" * 55)
print(f"Done.")
print(f"Reaches with data: {n_ok} / {len(query_reaches)}")
print(f"No SWOT data: {n_no_data}")
print(f"Errors: {n_error}")
print(f"Total observations: {len(all_features)}")

if error_log:
    print(f"\nError log (first 10):")
    for rid, msg in error_log[:10]:
        print(f"  reach {rid}: {msg}")

if n_ok == 0:
    print("\n No data returned for any reach.")

Reaches queried (width >= 100 m): 47
Skipped (too narrow): 93

Time range: 2025-08-01T00:00:00Z -> 2025-08-31T00:00:00Z
  0/47 | ok=0 no_data=0 error=0
 10/47 | ok=10 no_data=0 error=0
 20/47 | ok=20 no_data=0 error=0
 30/47 | ok=30 no_data=0 error=0
 40/47 | ok=39 no_data=0 error=1
-------------------------------------------------------
Done.
Reaches with data: 46 / 47
No SWOT data: 0
Errors: 1
Total observations: 199

Error log (first 10):
  reach 46219900396: 400 Bad Request: {'error': '400: Results with the specified Feature ID 46219900396 were not found'}


In [ ]:
# Parse results into GeoDataFrame and save

if not all_features:
    print("No features collected. Check time range and reach IDs.")
else:
    # Build GeoDataFrame from GeoJSON features
    swot_gdf = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:4326")

    print(f"GeoDataFrame shape : {swot_gdf.shape}")
    print(f"Columns: {swot_gdf.columns.tolist()}")
    print(f"CRS: {swot_gdf.crs}")

    # Convert time string to datetime for an easier analysis:
    if "time_str" in swot_gdf.columns:
        swot_gdf["time_str"] = pd.to_datetime(
            swot_gdf["time_str"], errors="coerce", utc=True
        )

    # Convert numeric columns --> force float dtype explicitly
    # pd.to_numeric alone sometimes leaves pandas StringArray dtype, 
    # so casting to float64 directly to ensure numeric comparison works.
    for col in ["wse", "slope", "width"]:
        if col in swot_gdf.columns:
            swot_gdf[col] = pd.to_numeric(
                swot_gdf[col], errors="coerce"
            ).astype("float64")

    SWOT_NODATA = -999999999999
    for col in ["wse", "slope", "width"]:
        if col in swot_gdf.columns:
            swot_gdf[col] = swot_gdf[col].where(swot_gdf[col] > SWOT_NODATA)
    print("\nBasic statistics:")

    num_cols = [c for c in ["wse", "slope", "width"] if c in swot_gdf.columns]
    print(swot_gdf[num_cols].describe().round(3))

    print(f"\nObservations per reach (distribution):")
    print(swot_gdf.groupby("reach_id").size().describe())

    # Save to GeoPackage
    # os.makedirs(DATA_PROCESSED, exist_ok=True)
    # swot_gdf.to_file(OUT_SWOT_HYDROCRON, driver="GPKG")
    # print(f"\nSaved: {OUT_SWOT_HYDROCRON}")

GeoDataFrame shape : (199, 11)
Columns: ['geometry', 'reach_id', 'time_str', 'wse', 'slope', 'width', 'p_dist_out', 'wse_units', 'slope_units', 'width_units', 'p_dist_out_units']
CRS: EPSG:4326

Basic statistics:
            wse   slope     width
count   148.000  94.000    98.000
mean   1349.420   0.004   402.714
std     786.592   0.002   438.221
min     403.200  -0.002     0.000
25%     723.699   0.003   181.000
50%    1258.199   0.003   272.837
75%    1661.500   0.004   472.918
max    3741.613   0.012  3082.287

Observations per reach (distribution):
count    46.000000
mean      4.326087
std       0.700931
min       3.000000
25%       4.000000
50%       4.000000
75%       5.000000
max       6.000000
dtype: float64


---
---
### 0.1.2 | earthaccess – Download SWOT granules

Downloads the actual SWOT Shapefile granules (one file = one satellite pass over a
continent). Files are filtered spatially by bounding box and temporally by date range.

**downloadable in this section:**
- Raster products (`L2_HR_Raster`)
- Full Shapefile attributes (not just the Hydrocron field subset)

**!!!Warning:!!!** Each SWOT granule covers an entire continent pass and can be **several
hundred MB to several GB**. After download the data is clipped to the AOI.

**NASA Earthdata Login required.** acc at: https://urs.earthdata.nasa.gov

References:
- earthaccess: https://earthaccess.readthedocs.io
- SWOT collections: https://podaac.github.io/tutorials/quarto_text/SWOT.html
- PO.DAAC Cookbook: https://podaac.github.io/tutorials

To-Do:
- [] create function for the download

In [39]:
# EARTHACCESS: authenticate with NASA Earthdata Login

# earthaccess.login() will:
#   Try to use a stored .netrc file silently
#   If not found: prompt for username + password interactively

auth = earthaccess.login(strategy="interactive", persist=True)
print(f"Login successful: {auth.authenticated}")

Login successful: True


In [42]:
# EARTHACCESS: search for granules

# NOTE: The bounding box filters which satellite passes intersect
# the AOI, NOT which features within the pass are inside it.
# Each granule still covers a full continent pass.
# !!!AOI-clipping happens in-memory in the next cell.!!

print(f"Searching for: {EARTHACCESS_COLLECTION}")
print(f"Bounding box: {BBOX}")
print(f"Time range: {START_TIME} to {END_TIME}")

results = earthaccess.search_data(
    short_name = EARTHACCESS_COLLECTION,
    bounding_box = BBOX,
    temporal = (START_TIME, END_TIME),
    count = -1 # -1 = return ALL matching granules
)

print(f"\nGranules found: {len(results)}")

# Show time span of first and last granule
if results:
    def granule_time(g):
        return g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime']

    print(f"First: {granule_time(results[0])}")
    print(f"Last : {granule_time(results[-1])}")

    # Estimate data volume
    est_mb_per_granule = 80 # rough estimation
    est_total_gb = len(results) * est_mb_per_granule/ 1000
    print(f"\nEstimated download size (if all downloaded): ca. {est_total_gb:.0f} GB")

Searching for: SWOT_L2_HR_RiverSP_reach_D
Bounding box: (np.float64(71.70473927538201), np.float64(40.854350089981835), np.float64(78.29815859096546), np.float64(42.36584381485357))
Time range: 2025-08-01T00:00:00Z to 2025-08-31T00:00:00Z

Granules found: 42
First: 2025-08-01T05:35:12.755Z
Last : 2025-08-27T02:27:02.793Z

Estimated download size (if all downloaded): ca. 3 GB


c:\Users\Duck\anaconda3\envs\river_egg\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


In [ ]:
# NOTE: Should create function for this but might be useless with my plan to do all this stuff easier and less....overloaded

# stream granules in-memory, clip to AOI, save
# No files are downloaded to disk. Each granule ZIP is opened
# directly via an authenticated HTTP stream, the Shapefile is
# read into memory, clipped to the AOI, then discarded.
# Only the merged AOI-clipped result is saved.
#
# GRANULE_STRIDE : 1 = every pass, 3 = ca. every third pass (ca. monthly)
# MAX_GRANULES   : hard cap for testing.... set to None for full run
GRANULE_STRIDE = 1
MAX_GRANULES   = NONE

# AOI bounding box for spatial clip (uses SWORD clip extent)
aoi_bounds = sword.total_bounds  #(minx, miny, maxx, maxy)
aoi_poly   = sword.dissolve().convex_hull  # tighter than bbox, optional

# Build the list of granules to process
process_list = results[::GRANULE_STRIDE]
if MAX_GRANULES:
    process_list = process_list[:MAX_GRANULES]

print(f"Granules to process : {len(process_list)} of {len(results)}")
print(f"Stride: {GRANULE_STRIDE}")
print(f"AOI bounds: {aoi_bounds.round(4)}")

# Open an authenticated session for streaming
# earthaccess.get_requests_https_session() returns a requests.Session
# with NASA EDL credentials pre-configured.
session = earthaccess.get_requests_https_session()

all_clipped= []   #list of clipped GeoDataFrames
n_ok = 0
n_empty = 0
n_error = 0

for i, granule in enumerate(process_list):
    if i % 50 == 0:
        print(f"{i:4d}/{len(process_list)} | ok={n_ok} empty={n_empty} error={n_error}")

    # Get the direct HTTPS download URL for this granule
    try:
        urls = granule.data_links(access="external")
        if not urls:
            # Fallback: extract URL from umm metadata directly
            related = granule['umm'].get('RelatedUrls', [])
            urls = [
                r['URL'] for r in related
                if r.get('Type') == 'GET DATA' and r['URL'].endswith('.zip')
            ]
        if not urls:
            n_error += 1
            continue
        zip_url = urls[0]
    except Exception as e:
        n_error += 1
        continue

    try:
        # Stream the ZIP into memory (no disk write)
        import io
        response = session.get(zip_url, timeout=60)
        response.raise_for_status()

        zip_buffer = io.BytesIO(response.content)

        # Find the Shapefile inside the ZIP
        with zipfile.ZipFile(zip_buffer) as zf:
            shp_names = [n for n in zf.namelist() if n.endswith('.shp')]
            if not shp_names:
                n_empty += 1
                continue

            # Read shapefile directly from the in-memory ZIP
            # Reset buffer so geopandas can re-read from it
            zip_buffer.seek(0)

        # geopandas zip:// URI syntax works with BytesIO via a temp path trick
        # Write ZIP to a small NamedTemporaryFile, read, then delete
        import tempfile
        with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmp:
            tmp.write(zip_buffer.getvalue())
            tmp_path = tmp.name

        try:
            gdf = gpd.read_file(f"zip://{tmp_path}!{shp_names[0]}")
        finally:
            os.unlink(tmp_path)   # delete temp file immediately after reading

        # Clip to AOI bounding box
        clipped = gdf.cx[
            aoi_bounds[0]:aoi_bounds[2],
            aoi_bounds[1]:aoi_bounds[3]
        ]

        if not clipped.empty:
            # Add observation timestamp from granule metadata
            clipped = clipped.copy()
            clipped["granule_time"] = granule['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime']
            all_clipped.append(clipped)
            n_ok += 1
        else:
            n_empty += 1

    except Exception as e:
        n_error += 1
        if n_error <= 5:
            print(f"Error on granule {i}: {e}")

print("-" * 55)
print(f"Done.")
print(f"Granules with AOI data : {n_ok}")
print(f"Granules empty (no AOI hit) : {n_empty}")
print(f"Errors: {n_error}")

# Merge and save
if all_clipped:
    swot_ea_gdf = gpd.GeoDataFrame(
        pd.concat(all_clipped, ignore_index=True),
        crs = all_clipped[0].crs
    )

    print(f"\nTotal observations: {len(swot_ea_gdf)}")
    print(f"Columns: {swot_ea_gdf.columns.tolist()}")

    # Convert time column
    swot_ea_gdf["granule_time"] = pd.to_datetime(
        swot_ea_gdf["granule_time"], utc=True
    )

    out_ea_gpkg = os.path.join(
        DATA_PROCESSED,
        f"swot_earthaccess_{EARTHACCESS_COLLECTION}_aoi_clip.gpkg"
    )
    os.makedirs(DATA_PROCESSED, exist_ok=True)
    swot_ea_gdf.to_file(out_ea_gpkg, driver="GPKG")
    print(f"\nSaved: {out_ea_gpkg}")
else:
    print("\nNo AOI features found across all processed granules.")
    print("Possible causes:")
    print("- SWOT did not observe this AOI in the selected time range")
    print("- River reaches are narrower than the SWOT 100 m threshold")
    print("- Bounding box does not intersect SWOT pass coverage")


Granules to process : 10 of 42
Stride: 1
AOI bounds: [71.7547 40.9044 78.2482 42.3158]
   0/10 | ok=0 empty=0 error=0
-------------------------------------------------------
Done.
Granules with AOI data : 5
Granules empty (no AOI hit) : 5
Errors: 0

Total observations: 210
Columns: ['reach_id', 'time', 'time_tai', 'time_str', 'p_lat', 'p_lon', 'river_name', 'wse', 'wse_u', 'wse_r_u', 'wse_c', 'wse_c_u', 'slope', 'slope_u', 'slope_r_u', 'slope2', 'slope2_u', 'slope2_r_u', 'width', 'width_u', 'width_c', 'width_c_u', 'area_total', 'area_tot_u', 'area_detct', 'area_det_u', 'area_wse', 'd_x_area', 'd_x_area_u', 'layovr_val', 'node_dist', 'loc_offset', 'xtrk_dist', 'dschg_c', 'dschg_c_u', 'dschg_csf', 'dschg_c_q', 'dschg_gc', 'dschg_gc_u', 'dschg_gcsf', 'dschg_gc_q', 'dschg_m', 'dschg_m_u', 'dschg_msf', 'dschg_m_q', 'dschg_gm', 'dschg_gm_u', 'dschg_gmsf', 'dschg_gm_q', 'dschg_b', 'dschg_b_u', 'dschg_bsf', 'dschg_b_q', 'dschg_gb', 'dschg_gb_u', 'dschg_gbsf', 'dschg_gb_q', 'dschg_h', 'dschg_h_